# Kaggle 3m Testing Pipeline: UNet++ Mask Prediction + GAN Volume Reconstruction

This notebook tests full-volume reconstruction under **low-dose (sparse known slices)** constraints.

Pipeline:
1. Load one patient volume from `archive/kaggle_3m` (2D TIFF stack).
2. Predict segmentation masks slice-by-slice using **UNet++ checkpoint**.
3. Reconstruct full volume using **GAN generator** with sparse known slices.
4. Evaluate quality (PSNR, SSIM, MSE) on all slices and unknown-only slices.
5. Export qualitative GIFs.

In [ ]:
from pathlib import Path
import random
import re
from typing import Dict, List, Tuple

import cv2
import imageio.v2 as imageio
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
else:
    DEVICE = torch.device('cpu')

print('Device:', DEVICE)

CFG = {
    'data_root': '/Volumes/T7/ORACLE project/archive/kaggle_3m',
    'img_size': 256,
    'patient_id': None,
    'mask_threshold': 0.50,
    'n_refine_passes': 2,
    'gif_fps': 8,
    'out_dir': 'assets/recon_media_kaggle3m',
    'candidate_unet_ckpts': [
        '/Users/enricotazzer/Desktop/ORACLE project/models/best_unet_plusplus.pth',
        '/Volumes/T7/ORACLE project/models/best_unet_plusplus.pth',
    ],
    'candidate_gan_ckpts': [
        '/Users/enricotazzer/Desktop/ORACLE project/models/best_gan_gen.pth',
        '/Volumes/T7/ORACLE project/models/best_gan_gen.pth',
    ],
}
print(CFG)

In [ ]:
def first_existing(paths: List[str]) -> Path:
    for p in paths:
        pp = Path(p)
        if pp.exists():
            return pp
    raise FileNotFoundError(f'None of these paths exists: {paths}')


def parse_slice_index(path: Path) -> int:
    name = path.stem
    if name.endswith('_mask'):
        name = name[:-5]
    m = re.search(r'(\d+)$', name)
    if m is None:
        raise ValueError(f'Could not parse slice index from {path.name}')
    return int(m.group(1))


def discover_patients(data_root: Path) -> List[Path]:
    return sorted([p for p in data_root.iterdir() if p.is_dir() and p.name.startswith('TCGA_')])


def load_patient_volume(patient_dir: Path) -> Dict[str, np.ndarray]:
    img_files = sorted(
        [p for p in patient_dir.glob('*.tif') if not p.name.endswith('_mask.tif')],
        key=parse_slice_index
    )
    if not img_files:
        raise RuntimeError(f'No image slices found in {patient_dir}')

    rgb_slices = []
    gray_slices = []
    for p in img_files:
        img = cv2.imread(str(p), cv2.IMREAD_COLOR)
        if img is None:
            raise RuntimeError(f'Failed to read {p}')
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (CFG['img_size'], CFG['img_size']), interpolation=cv2.INTER_LINEAR)
        img_f = img.astype(np.float32) / 255.0
        gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY).astype(np.float32) / 255.0
        rgb_slices.append(img_f)
        gray_slices.append(gray)

    rgb_vol = np.stack(rgb_slices, axis=2)     # [H,W,3,Z]
    t1n_vol = np.stack(gray_slices, axis=2)     # [H,W,Z]
    return {
        'rgb': rgb_vol.astype(np.float32),
        't1n': t1n_vol.astype(np.float32),
        'slice_files': img_files,
    }


DATA_ROOT = Path(CFG['data_root'])
if not DATA_ROOT.exists():
    raise FileNotFoundError(f'Data root not found: {DATA_ROOT}')

patients = discover_patients(DATA_ROOT)
if not patients:
    raise RuntimeError('No patient folders found in kaggle_3m.')

if CFG['patient_id'] is None:
    patient_dir = patients[0]
else:
    patient_dir = DATA_ROOT / CFG['patient_id']
    if not patient_dir.exists():
        raise FileNotFoundError(f'Patient folder not found: {patient_dir}')

patient = load_patient_volume(patient_dir)
print('Selected patient:', patient_dir.name)
print('RGB volume shape [H,W,3,Z]:', patient['rgb'].shape)
print('t1n volume shape [H,W,Z]:', patient['t1n'].shape)

In [ ]:
def minmax_norm_2d(x: np.ndarray, eps: float = 1e-8) -> np.ndarray:
    x = x.astype(np.float32)
    x_min = float(np.nanmin(x))
    x_max = float(np.nanmax(x))
    if x_max - x_min < eps:
        return np.zeros_like(x, dtype=np.float32)
    return (x - x_min) / (x_max - x_min + eps)


def build_unetpp_model(in_channels: int) -> nn.Module:
    try:
        import segmentation_models_pytorch as smp
    except Exception as ex:
        raise ImportError(
            'segmentation_models_pytorch is required. Install with: pip install segmentation-models-pytorch'
        ) from ex

    model = smp.UnetPlusPlus(
        encoder_name='resnet34',
        encoder_weights=None,
        in_channels=in_channels,
        classes=1,
        activation=None,
    )
    return model


def infer_unet_input_channels(state_dict: Dict[str, torch.Tensor]) -> int:
    candidates = [
        'encoder.conv1.weight',
        'module.encoder.conv1.weight',
        'encoder._conv_stem.weight',
        'module.encoder._conv_stem.weight',
    ]
    for k in candidates:
        if k in state_dict:
            return int(state_dict[k].shape[1])
    return 4


def adapt_rgb_to_unet_input(rgb_slice: np.ndarray, in_channels: int) -> np.ndarray:
    # rgb_slice: [H,W,3] in [0,1]
    gray = cv2.cvtColor((rgb_slice * 255.0).astype(np.uint8), cv2.COLOR_RGB2GRAY).astype(np.float32) / 255.0
    if in_channels == 1:
        x = gray[..., None]
    elif in_channels == 3:
        x = rgb_slice
    elif in_channels == 4:
        x = np.concatenate([rgb_slice, gray[..., None]], axis=-1)
    else:
        chans = [gray[..., None] for _ in range(in_channels)]
        x = np.concatenate(chans, axis=-1)
    return x.astype(np.float32)


@torch.no_grad()
def predict_mask_volume_unetpp(model: nn.Module, rgb_vol: np.ndarray, in_channels: int, thr: float) -> np.ndarray:
    # rgb_vol: [H,W,3,Z]
    model.eval()
    h, w, _, z_count = rgb_vol.shape
    probs = np.zeros((h, w, z_count), dtype=np.float32)

    for z in range(z_count):
        x = adapt_rgb_to_unet_input(rgb_vol[:, :, :, z], in_channels)
        x_t = torch.from_numpy(np.transpose(x, (2, 0, 1))).unsqueeze(0).to(DEVICE)

        # Simple TTA: horizontal flip average
        logit1 = model(x_t)
        logit2 = torch.flip(model(torch.flip(x_t, dims=[3])), dims=[3])
        prob = torch.sigmoid((logit1 + logit2) / 2.0).squeeze().float().cpu().numpy()
        probs[:, :, z] = prob.astype(np.float32)

    binary = (probs >= thr).astype(np.float32)
    return binary


UNET_CKPT = first_existing(CFG['candidate_unet_ckpts'])
print('UNet++ checkpoint:', UNET_CKPT)

unet_ckpt = torch.load(str(UNET_CKPT), map_location=DEVICE)
if isinstance(unet_ckpt, dict) and 'state_dict' in unet_ckpt:
    unet_sd = unet_ckpt['state_dict']
else:
    unet_sd = unet_ckpt

unet_sd = {k.replace('module.', '', 1) if k.startswith('module.') else k: v for k, v in unet_sd.items()}
unet_in_ch = infer_unet_input_channels(unet_sd)
unet_model = build_unetpp_model(unet_in_ch).to(DEVICE)
missing, unexpected = unet_model.load_state_dict(unet_sd, strict=False)
print(f'UNet++ input channels inferred: {unet_in_ch}')
print(f'UNet++ load_state_dict -> missing={len(missing)} unexpected={len(unexpected)}')

pred_mask_vol = predict_mask_volume_unetpp(
    unet_model,
    patient['rgb'],
    in_channels=unet_in_ch,
    thr=CFG['mask_threshold'],
)
print('Predicted mask volume shape:', pred_mask_vol.shape)
print('Predicted mask occupancy ratio:', float(pred_mask_vol.mean()))

In [ ]:
class ConvBlock2D(nn.Module):
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


class UNetDecoder2D(nn.Module):
    def __init__(self, in_ch: int = 96, out_ch: int = 1):
        super().__init__()
        self.enc1 = ConvBlock2D(in_ch, 128)
        self.pool = nn.MaxPool2d(2)
        self.enc2 = ConvBlock2D(128, 256)
        self.bottleneck = ConvBlock2D(256, 512)
        self.up2 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.dec2 = ConvBlock2D(512, 256)
        self.up1 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec1 = ConvBlock2D(256, 128)
        self.head = nn.Conv2d(128, out_ch, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        b = self.bottleneck(self.pool(e2))
        d2 = self.up2(b)
        d2 = self.dec2(torch.cat([d2, e2], dim=1))
        d1 = self.up1(d2)
        d1 = self.dec1(torch.cat([d1, e1], dim=1))
        return torch.sigmoid(self.head(d1))


class SliceCNN2D(nn.Module):
    def __init__(self, in_ch: int = 5, out_ch: int = 96):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, 48, 3, padding=1, bias=False),
            nn.BatchNorm2d(48),
            nn.ReLU(inplace=True),
            nn.Conv2d(48, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class Fast2p5D(nn.Module):
    def __init__(self, in_ch: int = 5, feat_ch: int = 96, out_ch: int = 1):
        super().__init__()
        self.slice_cnn = SliceCNN2D(in_ch=in_ch, out_ch=feat_ch)
        self.depth_attn = nn.Sequential(
            nn.Linear(feat_ch, feat_ch // 2),
            nn.ReLU(inplace=True),
            nn.Linear(feat_ch // 2, 1),
        )
        self.decoder = UNetDecoder2D(in_ch=feat_ch, out_ch=out_ch)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, c, d, h, w = x.shape
        feats = []
        for di in range(d):
            feats.append(self.slice_cnn(x[:, :, di, :, :]))
        feats = torch.stack(feats, dim=1)
        depth_desc = feats.mean(dim=(3, 4))
        logits = self.depth_attn(depth_desc).squeeze(-1)
        alpha = torch.softmax(logits, dim=1)[:, :, None, None, None]
        fused = (feats * alpha).sum(dim=1)
        return self.decoder(fused)


def load_generator(ckpt_path: Path) -> nn.Module:
    g = Fast2p5D(in_ch=5, feat_ch=96, out_ch=1).to(DEVICE)
    ckpt = torch.load(str(ckpt_path), map_location=DEVICE)

    if isinstance(ckpt, dict):
        if 'model_state_dict' in ckpt:
            sd = ckpt['model_state_dict']
        elif 'generator_state_dict' in ckpt:
            sd = ckpt['generator_state_dict']
        else:
            sd = ckpt
    else:
        sd = ckpt

    sd = {k.replace('module.', '', 1) if k.startswith('module.') else k: v for k, v in sd.items()}
    missing, unexpected = g.load_state_dict(sd, strict=False)
    print(f'Loaded GAN generator from {ckpt_path}')
    print(f'GAN load_state_dict -> missing={len(missing)} unexpected={len(unexpected)}')
    return g


GAN_CKPT = first_existing(CFG['candidate_gan_ckpts'])
G = load_generator(GAN_CKPT)
G.eval()

In [ ]:
def make_known_mask_n_uniform(z_count: int, n_keep: int) -> np.ndarray:
    n_keep = int(max(1, min(n_keep, z_count)))
    idx = np.unique(np.linspace(0, z_count - 1, num=n_keep, dtype=int))
    known = np.zeros(z_count, dtype=bool)
    known[idx] = True
    return known


def make_known_mask_random_fraction(z_count: int, frac: float, seed: int) -> np.ndarray:
    rng = np.random.default_rng(seed)
    n_keep = int(max(1, min(z_count, round(z_count * frac))))
    idx = np.sort(rng.choice(np.arange(z_count), size=n_keep, replace=False))
    known = np.zeros(z_count, dtype=bool)
    known[idx] = True
    return known


def init_t1n_state_from_known(t1n_gt: np.ndarray, known_mask: np.ndarray) -> np.ndarray:
    known_ids = np.where(known_mask)[0]
    if len(known_ids) == 0:
        raise ValueError('known_mask has no known slices')

    state = np.zeros_like(t1n_gt, dtype=np.float32)
    for z in range(t1n_gt.shape[2]):
        if known_mask[z]:
            state[:, :, z] = t1n_gt[:, :, z]
        else:
            nearest = int(known_ids[np.argmin(np.abs(known_ids - z))])
            state[:, :, z] = t1n_gt[:, :, nearest]
    return state


def build_context_slice(
    t1n_state: np.ndarray,
    t1n_gt: np.ndarray,
    pred_mask: np.ndarray,
    known_mask: np.ndarray,
    z_id: int,
    img_size: int,
) -> np.ndarray:
    z_max = t1n_gt.shape[2] - 1
    z = int(np.clip(z_id, 0, z_max))

    if known_mask[z]:
        base = minmax_norm_2d(t1n_gt[:, :, z])
    else:
        base = minmax_norm_2d(t1n_state[:, :, z])

    base = cv2.resize(base, (img_size, img_size), interpolation=cv2.INTER_LINEAR)
    msk = cv2.resize(pred_mask[:, :, z].astype(np.float32), (img_size, img_size), interpolation=cv2.INTER_LINEAR)

    # Kaggle has one image channel, so we copy base into GAN modality channels.
    c_t1n = base
    c_t1c = base
    c_t2w = base
    c_t2f = base
    c_msk = msk

    x = np.stack([c_t1n, c_t1c, c_t2w, c_t2f, c_msk], axis=0).astype(np.float32)
    return x


@torch.no_grad()
def gan_predict_with_tta(g_model: nn.Module, x: np.ndarray) -> np.ndarray:
    # x: [C,D,H,W]
    x_t = torch.from_numpy(x).unsqueeze(0).to(DEVICE)
    y1 = g_model(x_t)
    y2 = torch.flip(g_model(torch.flip(x_t, dims=[-1])), dims=[-1])
    y = ((y1 + y2) / 2.0).squeeze().cpu().numpy().astype(np.float32)
    return y


@torch.no_grad()
def reconstruct_full_volume_sparse(
    g_model: nn.Module,
    t1n_gt: np.ndarray,
    pred_mask: np.ndarray,
    known_mask: np.ndarray,
    img_size: int,
    n_refine_passes: int = 2,
) -> np.ndarray:
    h, w, z_count = t1n_gt.shape
    known_ids = np.where(known_mask)[0]
    if len(known_ids) == 0:
        raise ValueError('known_mask has no known slices')

    state = init_t1n_state_from_known(t1n_gt, known_mask).astype(np.float32)
    g_model.eval()

    for _ in range(n_refine_passes):
        fwd = state.copy()
        bwd = state.copy()

        for z in range(z_count):
            if known_mask[z]:
                fwd[:, :, z] = t1n_gt[:, :, z]
                continue
            z_ids = [z - 4, z - 3, z - 2, z - 1, z - 1]
            depth_items = [build_context_slice(fwd, t1n_gt, pred_mask, known_mask, zi, img_size) for zi in z_ids]
            x = np.transpose(np.stack(depth_items, axis=0), (1, 0, 2, 3)).astype(np.float32)
            pred = gan_predict_with_tta(g_model, x)
            fwd[:, :, z] = cv2.resize(pred, (w, h), interpolation=cv2.INTER_LINEAR)

        for z in range(z_count - 1, -1, -1):
            if known_mask[z]:
                bwd[:, :, z] = t1n_gt[:, :, z]
                continue
            z_ids = [z + 1, z + 2, z + 3, z + 4, z + 4]
            depth_items = [build_context_slice(bwd, t1n_gt, pred_mask, known_mask, zi, img_size) for zi in z_ids]
            x = np.transpose(np.stack(depth_items, axis=0), (1, 0, 2, 3)).astype(np.float32)
            pred = gan_predict_with_tta(g_model, x)
            bwd[:, :, z] = cv2.resize(pred, (w, h), interpolation=cv2.INTER_LINEAR)

        fused = np.empty_like(state, dtype=np.float32)
        for z in range(z_count):
            if known_mask[z]:
                fused[:, :, z] = t1n_gt[:, :, z]
                continue

            left_known = known_ids[known_ids <= z]
            right_known = known_ids[known_ids >= z]
            dl = (z - left_known[-1]) if len(left_known) > 0 else 1e9
            dr = (right_known[0] - z) if len(right_known) > 0 else 1e9

            wf = 1.0 / (dl + 1.0)
            wb = 1.0 / (dr + 1.0)
            s = wf + wb
            wf, wb = wf / s, wb / s

            fused[:, :, z] = wf * fwd[:, :, z] + wb * bwd[:, :, z]

        fused[:, :, known_mask] = t1n_gt[:, :, known_mask]
        state = fused

    return state.astype(np.float32)


def ssim_global_2d(x: np.ndarray, y: np.ndarray, c1: float = 0.01**2, c2: float = 0.03**2) -> float:
    x = x.astype(np.float32)
    y = y.astype(np.float32)
    mu_x = x.mean()
    mu_y = y.mean()
    var_x = ((x - mu_x) ** 2).mean()
    var_y = ((y - mu_y) ** 2).mean()
    cov_xy = ((x - mu_x) * (y - mu_y)).mean()
    num = (2 * mu_x * mu_y + c1) * (2 * cov_xy + c2)
    den = (mu_x**2 + mu_y**2 + c1) * (var_x + var_y + c2)
    return float(num / (den + 1e-12))


def compute_metrics(gt: np.ndarray, pred: np.ndarray, known_mask: np.ndarray) -> Dict[str, float]:
    mse_all = float(np.mean((pred - gt) ** 2))
    psnr_all = float(10.0 * np.log10(1.0 / max(mse_all, 1e-12)))
    ssim_all = float(np.mean([ssim_global_2d(gt[:, :, z], pred[:, :, z]) for z in range(gt.shape[2])]))

    unknown_ids = np.where(~known_mask)[0]
    if len(unknown_ids) > 0:
        gt_u = gt[:, :, unknown_ids]
        pr_u = pred[:, :, unknown_ids]
        mse_u = float(np.mean((pr_u - gt_u) ** 2))
        psnr_u = float(10.0 * np.log10(1.0 / max(mse_u, 1e-12)))
        ssim_u = float(np.mean([ssim_global_2d(gt[:, :, z], pred[:, :, z]) for z in unknown_ids]))
    else:
        mse_u, psnr_u, ssim_u = mse_all, psnr_all, ssim_all

    return {
        'mse_all': mse_all,
        'psnr_all': psnr_all,
        'ssim_all': ssim_all,
        'mse_unknown': mse_u,
        'psnr_unknown': psnr_u,
        'ssim_unknown': ssim_u,
        'known_slices': int(known_mask.sum()),
        'known_ratio': float(known_mask.mean()),
    }


def to_u8(x: np.ndarray, lo: float, hi: float) -> np.ndarray:
    if hi - lo < 1e-8:
        return np.zeros_like(x, dtype=np.uint8)
    y = np.clip((x - lo) / (hi - lo), 0.0, 1.0)
    return (255.0 * y).astype(np.uint8)


def export_gif(gt: np.ndarray, pred: np.ndarray, mask: np.ndarray, out_path: Path, label: str, fps: int = 8):
    out_path.parent.mkdir(parents=True, exist_ok=True)
    lo_g, hi_g = float(gt.min()), float(gt.max())
    lo_p, hi_p = float(pred.min()), float(pred.max())
    err = np.abs(gt - pred)
    hi_e = float(err.max() + 1e-8)

    frames = []
    for z in range(gt.shape[2]):
        g = np.flipud(to_u8(gt[:, :, z].T, lo_g, hi_g))
        p = np.flipud(to_u8(pred[:, :, z].T, lo_p, hi_p))
        e = np.flipud(to_u8(err[:, :, z].T, 0.0, hi_e))
        m = np.flipud((mask[:, :, z].T * 255.0).astype(np.uint8))

        g3 = np.stack([g, g, g], axis=-1)
        p3 = np.stack([p, p, p], axis=-1)
        e3 = np.stack([e, e, e], axis=-1)
        m3 = np.stack([m, m, m], axis=-1)

        frame = np.concatenate([g3, p3, e3, m3], axis=1)
        cv2.putText(frame, 'GT', (8, 24), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2, cv2.LINE_AA)
        cv2.putText(frame, 'Pred', (g3.shape[1] + 8, 24), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2, cv2.LINE_AA)
        cv2.putText(frame, 'AbsErr', (g3.shape[1] + p3.shape[1] + 8, 24), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2, cv2.LINE_AA)
        cv2.putText(frame, 'PredMask', (g3.shape[1] + p3.shape[1] + e3.shape[1] + 8, 24), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2, cv2.LINE_AA)
        cv2.putText(frame, f'{label} | z={z}', (8, 50), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 255, 255), 2, cv2.LINE_AA)
        frames.append(frame.astype(np.uint8))

    imageio.mimsave(out_path, frames, fps=fps, loop=0)
    return out_path

In [ ]:
t1n_gt = patient['t1n'].astype(np.float32)
z_count = t1n_gt.shape[2]

variant_known_masks = {
    'ultra_5_uniform': make_known_mask_n_uniform(z_count, n_keep=5),
    'ultra_8_uniform': make_known_mask_n_uniform(z_count, n_keep=8),
    'low_10pct_random': make_known_mask_random_fraction(z_count, frac=0.10, seed=SEED),
    'low_20pct_random': make_known_mask_random_fraction(z_count, frac=0.20, seed=SEED + 1),
}

out_dir = Path(CFG['out_dir'])
out_dir.mkdir(parents=True, exist_ok=True)

rows = []
gif_paths = []

for variant, known_mask in variant_known_masks.items():
    pred_vol = reconstruct_full_volume_sparse(
        g_model=G,
        t1n_gt=t1n_gt,
        pred_mask=pred_mask_vol,
        known_mask=known_mask,
        img_size=CFG['img_size'],
        n_refine_passes=CFG['n_refine_passes'],
    )

    metrics = compute_metrics(t1n_gt, pred_vol, known_mask)
    row = {'variant': variant}
    row.update(metrics)
    rows.append(row)

    gif_path = out_dir / f'{patient_dir.name}_{variant}_recon.gif'
    export_gif(t1n_gt, pred_vol, pred_mask_vol, gif_path, label=variant, fps=CFG['gif_fps'])
    gif_paths.append(gif_path)
    print(f'Saved: {gif_path}')

results_df = pd.DataFrame(rows).sort_values(by=['psnr_unknown', 'ssim_unknown'], ascending=False)
results_df

In [ ]:
summary_path = Path(CFG['out_dir']) / f'{patient_dir.name}_metrics.csv'
results_df.to_csv(summary_path, index=False)
print('Saved metrics table:', summary_path)

display_cols = [
    'variant',
    'known_slices',
    'known_ratio',
    'psnr_all',
    'ssim_all',
    'psnr_unknown',
    'ssim_unknown',
    'mse_unknown',
]
print(results_df[display_cols].to_string(index=False))

In [ ]:
z_mid = t1n_gt.shape[2] // 2
fig, ax = plt.subplots(1, 3, figsize=(13, 4))
ax[0].imshow(t1n_gt[:, :, z_mid], cmap='gray')
ax[0].set_title('GT middle slice')
ax[1].imshow(pred_mask_vol[:, :, z_mid], cmap='gray')
ax[1].set_title('UNet++ predicted mask')
ax[2].imshow(patient['rgb'][:, :, :, z_mid])
ax[2].set_title('Original RGB slice')
for a in ax:
    a.axis('off')
plt.tight_layout()
plt.show()

print('GIF exports:')
for p in gif_paths:
    print(p)